# Bài 02: Precision, Recall, F1-Score và Sự Đánh Đổi

**Mục tiêu bài học:**
- Hiểu sâu về mối quan hệ đánh đổi (trade-off) giữa Precision và Recall.
- Nắm được khái niệm và cách điều chỉnh ngưỡng quyết định (decision threshold) của mô hình.
- Học về đường cong Precision-Recall (Precision-Recall Curve) và ý nghĩa của nó.
- Hiểu và biết cách sử dụng F1-Score để có một chỉ số cân bằng giữa Precision và Recall.

## Ôn lại Precision và Recall

Ở bài trước, chúng ta đã học:
- **Precision (Độ chuẩn):** Trong số những gì được dự đoán là `Positive`, có bao nhiêu là `Positive` thật? Nó tập trung vào việc giảm thiểu **False Positives (FP)**.
  $$ \text{Precision} = \frac{\text{TP}}{\text{TP} + \text{FP}} $$
- **Recall (Độ phủ):** Trong số những gì là `Positive` thật, mô hình tìm ra được bao nhiêu? Nó tập trung vào việc giảm thiểu **False Negatives (FN)**.
  $$ \text{Recall} = \frac{\text{TP}}{\text{TP} + \text{FN}} $$

Một câu hỏi tự nhiên nảy sinh: Liệu chúng ta có thể có cả hai chỉ số này đều cao không? Câu trả lời là có, nhưng thường thì việc tăng chỉ số này sẽ làm giảm chỉ số kia. Đây được gọi là **sự đánh đổi Precision-Recall (Precision-Recall Trade-off)**.

## Ngưỡng Quyết Định (Decision Threshold)

Để hiểu được sự đánh đổi này, chúng ta cần biết cách các mô hình phân loại đưa ra quyết định.

Hầu hết các mô hình phân loại (như Logistic Regression, Random Forest,...) không trực tiếp đưa ra dự đoán `0` hay `1`. Thay vào đó, chúng đưa ra một **điểm số xác suất (probability score)** cho mỗi lớp. Ví dụ, với một mẫu dữ liệu, mô hình có thể dự đoán 80% khả năng là lớp `1` và 20% khả năng là lớp `0`.

Sau đó, chúng ta dùng một **ngưỡng quyết định (decision threshold)** để chuyển đổi xác suất này thành dự đoán cuối cùng. Mặc định, ngưỡng này thường là **0.5**.
- Nếu `xác suất >= 0.5`, dự đoán là `1` (Positive).
- Nếu `xác suất < 0.5`, dự đoán là `0` (Negative).

**Sự đánh đổi xảy ra khi chúng ta thay đổi ngưỡng này:**

- **Tăng ngưỡng (ví dụ: lên 0.8):**
  - Mô hình sẽ trở nên "khó tính" hơn khi dự đoán lớp `1`. Nó chỉ dự đoán `1` cho những trường hợp rất chắc chắn.
  - Kết quả: Số lượng False Positives (FP) giảm -> **Precision tăng**.
  - Tuy nhiên, nhiều trường hợp `Positive` thật sự nhưng có xác suất thấp hơn 0.8 sẽ bị dự đoán nhầm là `0`. Số lượng False Negatives (FN) tăng -> **Recall giảm**.

- **Giảm ngưỡng (ví dụ: xuống 0.3):**
  - Mô hình sẽ trở nên "dễ tính" hơn khi dự đoán lớp `1`.
  - Kết quả: Mô hình sẽ "vớt" được nhiều trường hợp `Positive` hơn, kể cả những trường hợp không chắc chắn lắm. Số lượng False Negatives (FN) giảm -> **Recall tăng**.
  - Tuy nhiên, việc này cũng làm tăng khả năng dự đoán nhầm các trường hợp `Negative` thành `Positive`. Số lượng False Positives (FP) tăng -> **Precision giảm**.

### Ví dụ với Python

Hãy huấn luyện một mô hình đơn giản và xem sự thay đổi của Precision/Recall khi thay đổi ngưỡng.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
from sklearn.metrics import precision_score, recall_score

# Tạo dữ liệu giả lập
X, y = make_classification(n_samples=1000, n_features=20, n_informative=2, 
                           n_redundant=10, n_classes=2, random_state=42)

# Chia dữ liệu
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Huấn luyện mô hình
model = LogisticRegression()
model.fit(X_train, y_train)

# Lấy xác suất dự đoán cho lớp 1
y_scores = model.predict_proba(X_test)[:, 1]

# 1. Với ngưỡng mặc định 0.5
y_pred_default = (y_scores >= 0.5).astype(int)
precision_default = precision_score(y_test, y_pred_default)
recall_default = recall_score(y_test, y_pred_default)
print(f"Ngưỡng 0.5 (Mặc định):")
print(f"- Precision: {precision_default:.4f}")
print(f"- Recall:    {recall_default:.4f}")

# 2. Tăng ngưỡng lên 0.8
y_pred_high_thresh = (y_scores >= 0.8).astype(int)
precision_high = precision_score(y_test, y_pred_high_thresh)
recall_high = recall_score(y_test, y_pred_high_thresh)
print(f"\nNgưỡng 0.8 (Tăng Precision):")
print(f"- Precision: {precision_high:.4f}")
print(f"- Recall:    {recall_high:.4f}")

# 3. Giảm ngưỡng xuống 0.3
y_pred_low_thresh = (y_scores >= 0.3).astype(int)
precision_low = precision_score(y_test, y_pred_low_thresh)
recall_low = recall_score(y_test, y_pred_low_thresh)
print(f"\nNgưỡng 0.3 (Tăng Recall):")
print(f"- Precision: {precision_low:.4f}")
print(f"- Recall:    {recall_low:.4f}")

Như bạn thấy, khi tăng ngưỡng, Precision tăng và Recall giảm. Ngược lại, khi giảm ngưỡng, Recall tăng và Precision giảm.

## Đường cong Precision-Recall (Precision-Recall Curve)

Thay vì thử từng ngưỡng một cách thủ công, chúng ta có thể trực quan hóa sự đánh đổi này trên một biểu đồ gọi là **Đường cong Precision-Recall**.

- Trục tung là Precision.
- Trục hoành là Recall.

Đường cong này được vẽ bằng cách tính Precision và Recall ở tất cả các ngưỡng có thể có. Mỗi điểm trên đường cong tương ứng với một giá trị ngưỡng.

**Ý nghĩa:**
- Một mô hình lý tưởng sẽ có đường cong đi thẳng từ góc dưới bên trái lên góc trên bên phải (Precision=1, Recall=1). Điều này trong thực tế gần như không thể.
- Một mô hình càng tốt, đường cong của nó càng gần góc trên bên phải.
- Đường cong giúp chúng ta chọn một ngưỡng phù hợp với bài toán. Ví dụ, nếu bài toán yêu cầu Recall tối thiểu 80%, bạn có thể tìm điểm trên đường cong có Recall=0.8 và xem Precision tương ứng là bao nhiêu, và ngưỡng nào cho ra kết quả đó.

In [ ]:
from sklearn.metrics import precision_recall_curve
import matplotlib.pyplot as plt

# Tính các giá trị cho đường cong
precisions, recalls, thresholds = precision_recall_curve(y_test, y_scores)

# Vẽ đường cong
plt.figure(figsize=(8, 6))
plt.plot(recalls, precisions, marker='.')
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.grid(True)
plt.show()

Đường cong cho thấy rõ khi Recall tăng (đi về bên phải), Precision có xu hướng giảm (đi xuống dưới).

## F1-Score: Cân bằng giữa Precision và Recall

Việc phải nhìn vào hai chỉ số Precision và Recall cùng lúc đôi khi khá bất tiện. Chúng ta thường muốn có một chỉ số duy nhất để so sánh các mô hình với nhau. **F1-Score** ra đời để giải quyết vấn đề này.

F1-Score là **trung bình điều hòa (harmonic mean)** của Precision và Recall.

$$ F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}} $$

**Tại sao lại dùng trung bình điều hòa?**
Không giống như trung bình cộng, trung bình điều hòa sẽ "trừng phạt" nặng nếu một trong hai chỉ số (Precision hoặc Recall) quá thấp. Một F1-Score cao chỉ có thể đạt được khi **cả hai** Precision và Recall đều cao.

Ví dụ:
- Precision = 1.0, Recall = 0.1 -> F1-Score ≈ 0.18 (rất thấp)
- Precision = 0.6, Recall = 0.5 -> F1-Score ≈ 0.55
- Precision = 0.9, Recall = 0.9 -> F1-Score = 0.9

**Khi nào nên dùng F1-Score?**
Khi bạn muốn tìm một sự cân bằng giữa Precision và Recall, và tầm quan trọng của FP và FN là tương đương nhau. Đây là chỉ số rất phổ biến trong các bài toán có dữ liệu mất cân bằng.

In [ ]:
from sklearn.metrics import f1_score

# Tính F1-score cho các ngưỡng đã thử
f1_default = f1_score(y_test, y_pred_default)
f1_high = f1_score(y_test, y_pred_high_thresh)
f1_low = f1_score(y_test, y_pred_low_thresh)

print(f"F1-Score với ngưỡng 0.5: {f1_default:.4f}")
print(f"F1-Score với ngưỡng 0.8: {f1_high:.4f}")
print(f"F1-Score với ngưỡng 0.3: {f1_low:.4f}")

### Tìm ngưỡng tối ưu cho F1-Score

Chúng ta có thể tìm ra ngưỡng quyết định nào sẽ cho F1-Score cao nhất.

In [ ]:
import numpy as np

# Tính F1-score cho tất cả các ngưỡng
# Lưu ý: thresholds từ precision_recall_curve không bao gồm ngưỡng cuối cùng, 
# nên precisions và recalls sẽ có nhiều hơn 1 phần tử.
f1_scores = 2 * (precisions * recalls) / (precisions + recalls)
# Bỏ qua giá trị cuối cùng của f1_scores (có thể là nan)
f1_scores = f1_scores[:-1]

# Tìm ngưỡng cho F1-score cao nhất
best_f1_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_f1_idx]
best_f1 = f1_scores[best_f1_idx]

print(f"F1-Score cao nhất là {best_f1:.4f} tại ngưỡng ~{best_threshold:.4f}")

## Tóm tắt

- **Precision và Recall** thường có mối quan hệ đánh đổi: tăng cái này làm giảm cái kia.
- Sự đánh đổi này được kiểm soát bằng cách thay đổi **ngưỡng quyết định (decision threshold)** của mô hình.
- **Đường cong Precision-Recall** là công cụ trực quan để phân tích sự đánh đổi này ở mọi ngưỡng.
- **F1-Score** là trung bình điều hòa của Precision và Recall, cung cấp một chỉ số duy nhất để đo lường hiệu quả khi cần cân bằng cả hai.

Trong bài học tiếp theo, chúng ta sẽ khám phá một công cụ đánh giá tổng quan khác rất mạnh mẽ là đường cong ROC và chỉ số AUC, đặc biệt hữu ích khi so sánh hiệu năng tổng thể của các mô hình khác nhau.